# 10 — LLM Report Validation

This notebook validates the **real LLM reporting layer** against the deterministic local renderer.

It does not retrain any anomaly detector.

Objectives:

- collect representative `LOW`, `MEDIUM`, and `HIGH` investigation packets
- generate the deterministic local report for each packet
- optionally generate an OpenAI API report for the same packet
- verify semantic invariants automatically
- compare local vs LLM wording without allowing the LLM to change upstream decisions
- save validation results for the portfolio

The LLM is evaluated as a **report generator**, not as a fraud detector.


## Imports and project path

In [ ]:
from pathlib import Path
import sys
import json
import os
import re
from getpass import getpass

import pandas as pd

# This notebook lives in notebooks/, so add the project root explicitly.
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


In [ ]:
from src.investigation import (
    contains_forbidden_ground_truth
)

from src.reporting import (
    generate_llm_report,
    local_report_from_packet,
    report_to_markdown
)

print("src imports OK")


## Paths

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results"
CLI_PATH = RESULTS_PATH / "cli"
INVESTIGATION_PATH = RESULTS_PATH / "investigations"
VALIDATION_PATH = RESULTS_PATH / "llm_validation"

VALIDATION_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Validation output:", VALIDATION_PATH)


## Load available investigation packets

In [ ]:
packets = []

# Packets produced by the CLI.
if CLI_PATH.exists():
    for path in sorted(
        CLI_PATH.glob(
            "transaction_*_packet.json"
        )
    ):
        with path.open(
            "r",
            encoding="utf-8"
        ) as f:
            packet = json.load(f)

        packet["_source_file"] = str(path)
        packets.append(packet)

# Packets exported by notebook 08.
top_packets_path = (
    INVESTIGATION_PATH
    / "top_100_investigation_packets.json"
)

if top_packets_path.exists():
    with top_packets_path.open(
        "r",
        encoding="utf-8"
    ) as f:
        exported_packets = json.load(f)

    for packet in exported_packets:
        packet = packet.copy()
        packet["_source_file"] = str(
            top_packets_path
        )
        packets.append(packet)

print("Packets loaded:", len(packets))


## Deduplicate packets

In [ ]:
packet_by_id = {}

for packet in packets:
    transaction_id = int(
        packet["transaction_id"]
    )

    packet_by_id[
        transaction_id
    ] = packet

packets = list(
    packet_by_id.values()
)

print(
    "Unique transactions:",
    len(packets)
)


## Ground-truth leakage guard

In [ ]:
for packet in packets:
    packet_for_check = {
        key: value
        for key, value in packet.items()
        if not key.startswith("_")
    }

    if contains_forbidden_ground_truth(
        packet_for_check
    ):
        raise ValueError(
            "Forbidden ground-truth field found "
            f"in transaction {packet['transaction_id']}"
        )

print(
    "All available packets pass the "
    "ground-truth leakage guard."
)


## Select one representative case per risk tier

In [ ]:
selected_packets = {}

for tier in [
    "LOW",
    "MEDIUM",
    "HIGH"
]:
    matches = [
        packet
        for packet in packets
        if packet["risk_tier"] == tier
    ]

    if matches:
        selected_packets[
            tier
        ] = matches[0]

print(
    "Selected tiers:",
    list(selected_packets)
)

for tier, packet in selected_packets.items():
    print(
        tier,
        "→ transaction",
        packet["transaction_id"]
    )


Representative transactions already validated manually in this project:

- LOW: transaction `483337`
- MEDIUM: transaction `483339`
- HIGH: transaction `483406`

If those packet files exist, the next cell forces the notebook to use them.


In [ ]:
PREFERRED_TRANSACTION_IDS = {
    "LOW": 483337,
    "MEDIUM": 483339,
    "HIGH": 483406,
}

for tier, transaction_id in (
    PREFERRED_TRANSACTION_IDS.items()
):
    if transaction_id in packet_by_id:
        selected_packets[tier] = (
            packet_by_id[
                transaction_id
            ]
        )

for tier, packet in selected_packets.items():
    print(
        tier,
        "→ transaction",
        packet["transaction_id"]
    )


## Require all three tiers

In [ ]:
missing_tiers = [
    tier
    for tier in [
        "LOW",
        "MEDIUM",
        "HIGH"
    ]
    if tier not in selected_packets
]

if missing_tiers:
    raise RuntimeError(
        "Missing representative packet(s) for: "
        + ", ".join(missing_tiers)
        + ". Generate them with the CLI first."
    )

print("LOW / MEDIUM / HIGH packets ready.")


## Generate deterministic local reports

In [ ]:
local_reports = {}

for tier, packet in (
    selected_packets.items()
):
    clean_packet = {
        key: value
        for key, value in packet.items()
        if not key.startswith("_")
    }

    local_reports[tier] = (
        local_report_from_packet(
            clean_packet
        )
    )

for tier, report in (
    local_reports.items()
):
    print(
        "\n",
        "=" * 70,
        "\n",
        tier,
        "\n",
        "=" * 70,
        sep=""
    )

    print(
        report_to_markdown(
            report
        )
    )


## Semantic validation helpers

In [ ]:
def flatten_report_text(report):
    """
    Flatten report text safely.

    Any missing/None field is converted to an empty string so the
    semantic validator never crashes on optional/empty LLM output.
    """

    def safe_text(value):
        if value is None:
            return ""
        return str(value)

    def safe_list(values):
        if values is None:
            return []

        if not isinstance(values, list):
            values = [values]

        return [
            safe_text(value)
            for value in values
            if value is not None
        ]

    parts = [
        safe_text(report.get("executive_summary")),
        safe_text(report.get("model_consensus")),
        safe_text(report.get("conclusion")),
        *safe_list(report.get("model_evidence")),
        *safe_list(report.get("contextual_evidence")),
        *safe_list(report.get("autoencoder_evidence")),
        *safe_list(report.get("recommended_analyst_checks")),
        *safe_list(report.get("limitations")),
    ]

    return " ".join(
        part
        for part in parts
        if part
    ).lower()


def has_affirmative_fraud_claim(text):
    """
    Detect explicit affirmative fraud claims while allowing safe
    negative/uncertain statements such as:

    'There is insufficient evidence to conclude that the transaction
    is fraudulent.'
    """

    if text is None:
        return False

    text = str(text).lower()

    sentences = re.split(
        r"[.!?]+",
        text
    )

    affirmative_patterns = [
        r"^\s*(the\s+)?transaction\s+is\s+fraudulent\b",
        r"^\s*(the\s+)?transaction\s+is\s+confirmed\s+fraud\b",
        r"^\s*fraud\s+is\s+confirmed\b",
        r"\bconfirmed\s+as\s+fraud\b",
        r"\bdefinitely\s+fraudulent\b",
        r"\bclearly\s+fraudulent\b",
        r"\bcertainly\s+fraudulent\b",
    ]

    negation_markers = [
        "not",
        "no evidence",
        "insufficient evidence",
        "does not",
        "do not",
        "cannot",
        "can't",
        "unable to",
        "doesn't establish",
        "cannot conclude",
        "not enough evidence",
        "does not establish",
        "cannot establish",
    ]

    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue

        if any(
            marker in sentence
            for marker in negation_markers
        ):
            continue

        for pattern in affirmative_patterns:
            if re.search(
                pattern,
                sentence
            ):
                return True

    return False


def validate_report_semantics(
    report,
    packet
):
    issues = []

    if report is None:
        return [
            "report is None"
        ]

    if not isinstance(
        report,
        dict
    ):
        return [
            "report is not a dictionary"
        ]

    text = flatten_report_text(
        report
    )

    # -------------------------
    # Required output fields
    # -------------------------

    required_fields = [
        "transaction_id",
        "risk_tier",
        "executive_summary",
        "model_evidence",
        "contextual_evidence",
        "autoencoder_evidence",
        "model_consensus",
        "recommended_analyst_checks",
        "limitations",
        "conclusion",
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in report
    ]

    if missing_fields:
        issues.append(
            "missing report fields: "
            + ", ".join(
                missing_fields
            )
        )

    # -------------------------
    # Identity invariants
    # -------------------------

    if (
        report.get("transaction_id")
        != packet.get("transaction_id")
    ):
        issues.append(
            "transaction_id changed"
        )

    if (
        report.get("risk_tier")
        != packet.get("risk_tier")
    ):
        issues.append(
            "risk_tier changed"
        )

    # -------------------------
    # Fraud-claim safety
    # -------------------------

    if has_affirmative_fraud_claim(
        text
    ):
        issues.append(
            "affirmative fraud claim detected"
        )

    # -------------------------
    # Probability misuse
    # -------------------------

    positive_probability_claims = [
        "fraud probability is",
        "probability of fraud is",
        "fraud probability:",
    ]

    for phrase in positive_probability_claims:
        if phrase in text:
            issues.append(
                "score interpreted as probability: "
                + phrase
            )

    # -------------------------
    # Risk-tier semantics
    # -------------------------

    tier = packet.get(
        "risk_tier"
    )

    if tier == "LOW":

        if (
            "priority analyst review"
            in text
        ):
            issues.append(
                "LOW case escalated to priority review"
            )

        if (
            "routine monitoring"
            not in text
        ):
            issues.append(
                "LOW case does not mention routine monitoring"
            )

    elif tier == "MEDIUM":

        if (
            "routine analyst review"
            not in text
        ):
            issues.append(
                "MEDIUM case missing routine analyst review"
            )

        if (
            "priority analyst review"
            in text
        ):
            issues.append(
                "MEDIUM case escalated to priority review"
            )

    elif tier == "HIGH":

        if (
            "priority analyst review"
            not in text
        ):
            issues.append(
                "HIGH case missing priority analyst review"
            )

    return issues


In [ ]:
# Quick validator sanity check before running the full validation.

for tier, report in local_reports.items():
    print(
        tier,
        "flattened text length:",
        len(flatten_report_text(report))
    )

print("Semantic validator helpers ready.")


## Validate local reports

In [ ]:
local_quality_rows = []

for tier, packet in (
    selected_packets.items()
):
    clean_packet = {
        key: value
        for key, value in packet.items()
        if not key.startswith("_")
    }

    report = local_reports[
        tier
    ]

    issues = validate_report_semantics(
        report,
        clean_packet
    )

    local_quality_rows.append({
        "renderer": "local",
        "risk_tier": tier,
        "transaction_id":
            packet["transaction_id"],
        "valid":
            len(issues) == 0,
        "issues":
            " | ".join(issues),
    })

local_quality = pd.DataFrame(
    local_quality_rows
)

local_quality


## OpenAI API configuration

In [ ]:
MODEL = os.getenv(
    "OPENAI_MODEL",
    "gpt-5.6-luna"
)

HAS_API_KEY = bool(
    os.getenv(
        "OPENAI_API_KEY"
    )
)

print("Model:", MODEL)
print(
    "OPENAI_API_KEY configured:",
    HAS_API_KEY
)


## Secure API-key input

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "OpenAI API key: "
    )

HAS_API_KEY = bool(
    os.getenv("OPENAI_API_KEY")
)

print(
    "OPENAI_API_KEY configured:",
    HAS_API_KEY
)


The key entered with `getpass()` is not displayed in the notebook output and is only stored in the current kernel environment.

Do not hard-code API keys in the notebook or commit them to Git.


## Generate real LLM reports

In [ ]:
# Change to True only when you intentionally
# want to make the three API calls.

RUN_LLM = True


In [ ]:
llm_reports = {}

if RUN_LLM:

    if not HAS_API_KEY:
        raise RuntimeError(
            "OPENAI_API_KEY is not configured."
        )

    for tier, packet in (
        selected_packets.items()
    ):
        clean_packet = {
            key: value
            for key, value in packet.items()
            if not key.startswith("_")
        }

        print(
            "Generating",
            tier,
            "report for transaction",
            packet["transaction_id"]
        )

        llm_reports[tier] = (
            generate_llm_report(
                clean_packet,
                model=MODEL,
            )
        )

else:

    print(
        "RUN_LLM=False → no API calls made."
    )


## Validate real LLM reports

In [ ]:
llm_quality_rows = []

for tier, report in (
    llm_reports.items()
):
    packet = selected_packets[
        tier
    ]

    clean_packet = {
        key: value
        for key, value in packet.items()
        if not key.startswith("_")
    }

    issues = validate_report_semantics(
        report,
        clean_packet
    )

    llm_quality_rows.append({
        "renderer": "llm",
        "risk_tier": tier,
        "transaction_id":
            packet["transaction_id"],
        "valid":
            len(issues) == 0,
        "issues":
            " | ".join(issues),
    })

llm_quality = pd.DataFrame(
    llm_quality_rows
)

llm_quality


## Compare local vs LLM reports

In [ ]:
comparison_rows = []

for tier in [
    "LOW",
    "MEDIUM",
    "HIGH"
]:
    local_report = local_reports[
        tier
    ]

    llm_report = llm_reports.get(
        tier
    )

    comparison_rows.append({
        "risk_tier":
            tier,

        "transaction_id":
            selected_packets[
                tier
            ][
                "transaction_id"
            ],

        "local_summary":
            local_report[
                "executive_summary"
            ],

        "llm_summary":
            (
                llm_report[
                    "executive_summary"
                ]
                if llm_report
                else None
            ),

        "local_conclusion":
            local_report[
                "conclusion"
            ],

        "llm_conclusion":
            (
                llm_report[
                    "conclusion"
                ]
                if llm_report
                else None
            ),
    })

comparison = pd.DataFrame(
    comparison_rows
)

comparison


## Save validation outputs

In [ ]:
local_quality.to_csv(
    VALIDATION_PATH
    / "local_report_validation.csv",
    index=False
)

comparison.to_csv(
    VALIDATION_PATH
    / "local_vs_llm_comparison.csv",
    index=False
)

if not llm_quality.empty:
    llm_quality.to_csv(
        VALIDATION_PATH
        / "llm_report_validation.csv",
        index=False
    )

for tier, report in (
    llm_reports.items()
):
    transaction_id = report[
        "transaction_id"
    ]

    with (
        VALIDATION_PATH
        / (
            f"{tier.lower()}_"
            f"{transaction_id}_llm_report.json"
        )
    ).open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            report,
            f,
            indent=2,
            ensure_ascii=False
        )

    with (
        VALIDATION_PATH
        / (
            f"{tier.lower()}_"
            f"{transaction_id}_llm_report.md"
        )
    ).open(
        "w",
        encoding="utf-8"
    ) as f:
        f.write(
            report_to_markdown(
                report
            )
        )

print(
    "Saved validation outputs to:",
    VALIDATION_PATH
)


## Final validation summary

In [ ]:
print("LOCAL REPORTS")
print(
    local_quality[
        [
            "risk_tier",
            "transaction_id",
            "valid",
            "issues"
        ]
    ].to_string(
        index=False
    )
)

if not llm_quality.empty:
    print("\nLLM REPORTS")
    print(
        llm_quality[
            [
                "risk_tier",
                "transaction_id",
                "valid",
                "issues"
            ]
        ].to_string(
            index=False
        )
    )


## Success criteria

### LOW
- keeps the upstream LOW tier
- does not request priority review
- recommends routine monitoring
- does not overstate weak Autoencoder reconstruction errors

### MEDIUM
- keeps the upstream MEDIUM tier
- recommends routine analyst review
- does not escalate to priority review

### HIGH
- keeps the upstream HIGH tier
- recommends priority analyst review
- may describe strong model consensus
- still does not declare fraud

### All tiers
- preserve transaction ID
- preserve risk tier
- never treat anomaly scores as fraud probabilities
- never introduce the hidden ground-truth label
- never invent unsupported evidence

The deterministic local renderer is the baseline. The LLM is useful only if it improves readability while respecting the same constraints.


## Next step

Once all six validation rows are `True`:

```text
LOCAL: LOW / MEDIUM / HIGH
LLM:   LOW / MEDIUM / HIGH
```

the reporting layer can be considered validated and the project can move to final GitHub/portfolio packaging.
